# Silver Validation

Validates, cleans and deduplicates employee data from the Bronze layer.

## Setup

In [0]:
%sql

USE CATALOG workspace;
USE SCHEMA people_data_silver;

## Load Bronze Data

In [0]:
bronze_df = spark.table(
    "people_data_bronze.employees_raw"
)
display(bronze_df)

In [0]:
bronze_df.printSchema()

## Type Casting

In [0]:
from pyspark.sql.functions import col
silver_df = (
    bronze_df
    .withColumn("employee_id", col("employee_id").cast("int"))
    .withColumn("department_id", col("department_id").cast("int"))
    .withColumn("salary", col("salary").cast("int"))
)

In [0]:
silver_df.select(
    "employee_id",
    "department_id",
    "salary"
).printSchema()

## Date Parsing

In [0]:
silver_df.select("hire_date").show(500, truncate=False)

In [0]:
from pyspark.sql.functions import try_to_timestamp, lit

silver_df = silver_df.withColumn(
    "hire_date_parsed",
    try_to_timestamp(
        col("hire_date"),
        lit("yyyy-MM-dd HH:mm:ss")
    )
)

In [0]:
silver_df.select(
    "hire_date",
    "hire_date_parsed"
).show(50, truncate=False)

In [0]:
silver_df = silver_df.withColumn(
    "updated_at",
    try_to_timestamp(
        col("updated_at"),
        lit("yyyy-MM-dd HH:mm:ss")
    )   
)

In [0]:
silver_df.select("updated_at").show(100, truncate=False)

In [0]:
silver_df.select(
    "updated_at"
).printSchema()

## Data Quality Checks

In [0]:
from pyspark.sql.functions import when

silver_df = silver_df.withColumn(
    "salary_error",
    when(col("salary") <= 0,
         "INVALID_SALARY"
    ).otherwise(None)
)    

In [0]:
silver_df.select(
    "employee_id",
    "salary",
    "salary_error"
).show(20, truncate=False)

In [0]:
display(silver_df)

In [0]:
silver_df = silver_df.withColumn(
    "status_error",
    when(
        col("status").isNull()
    | ~col("status").isin("ACTIVE","INACTIVE"), 
          "INVALID_STATUS"
         ).otherwise(None)
)

In [0]:
display(silver_df)

In [0]:
from pyspark.sql.functions import when, trim
silver_df = silver_df.withColumn(
    "country_error",
    when(
        col("country").isNull()
        | (trim(col("country")) == ""),
        "EMPTY_COUNTRY"
        ).otherwise(None)
    )

In [0]:
silver_df.select(
    "employee_id",
    "country",
    "country_error"
).show(50, truncate=False)

In [0]:
email_pattern = r"\A[^@\s]+@[^@\s]+\.[^@\s]+\z"

silver_df = silver_df.withColumn(
    "email_error",
    when(
        col("email").isNull()
        | (trim(col("email")) == ""),
        "EMPTY_EMAIL")
    .when(
        ~col("email").rlike(email_pattern),
        "INVALID_EMAIL"
    ).otherwise(None)
    )

In [0]:
display(silver_df)

In [0]:
silver_df = silver_df.withColumn(
    "hire_date_error",
    when(
        col("hire_date").isNull()
        | (trim(col("hire_date")) == ""),
        "EMPTY_HIRE_DATE"
    ).when(
        col("hire_date").isNotNull()
           & col("hire_date_parsed").isNull(),
           "INVALID_HIRE_DATE"
    ).otherwise(None))

In [0]:
silver_df.select(
    "hire_date_error"
).filter(
    col("hire_date_error").isNotNull()
).show(50, truncate=False
)

In [0]:
silver_df.groupBy("hire_date_error").count().show()

In [0]:
silver_df.filter(
    col("hire_date_error").isNotNull()
).groupBy(
    "hire_date_error"
).count().show()

## Deduplication

In [0]:
duplicates_df = silver_df.groupBy("employee_id").count().filter(
    col("count") > 1
)
duplicates_df.show()

In [0]:
duplicates_df.count()

In [0]:
silver_df.printSchema()

In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

employee_window = Window.partitionBy("employee_id").orderBy(col("updated_at").desc())

In [0]:
silver_df = silver_df.withColumn(
    "row_num",
    row_number().over(employee_window)
)

In [0]:
latest_df = silver_df.filter(
    col("row_num") == 1 
)


In [0]:
excluded_df = silver_df.filter(
    col("row_num") > 1
)
print(excluded_df.count())
excluded_df.show()

In [0]:
print("Wszystkie rekordy:", silver_df.count())
print("Najnowsze rekordy:", latest_df.count())
print("Starsze duplikaty:", excluded_df.count())

## Error Reasons

In [0]:
from pyspark.sql.functions import concat_ws

latest_df = latest_df.withColumn(
    "error_reasons",
    concat_ws(
        ", ",
        col("salary_error"),
        col("status_error"),
        col("country_error"),
        col("email_error"),
        col("hire_date_error")
    )
)

In [0]:
latest_df.select(
    "employee_id",
    "salary_error",
    "status_error",
    "country_error",
    "email_error",
    "hire_date_error",
    "error_reasons"
).filter(
    col("error_reasons") != ""
).show(50, truncate=False)

## Split Clean and Rejected Records

In [0]:
clean_df = latest_df.filter(
    col("error_reasons") == ""
)

In [0]:
display(clean_df)

In [0]:
rejected_df = latest_df.filter(
    col("error_reasons") != ""
)


In [0]:
display(rejected_df)

## Reconciliation Check

In [0]:
source_count = silver_df.count()
latest_count = latest_df.count()
excluded_count = excluded_df.count()
clean_count = clean_df.count()
rejected_count = rejected_df.count()

print("Rekordy źródłowe:", source_count)
print("Po deduplikacji:", latest_count)
print("Starsze duplikaty:", excluded_count)
print("Clean:", clean_count)
print("Rejected:", rejected_count)

print("Bilans po deduplikacji:", latest_count + excluded_count == source_count)
print("Bilans Silver:", clean_count + rejected_count == latest_count)
print("Bilans końcowy:", clean_count + rejected_count + excluded_count == source_count)

## Prepare Clean Dataset

In [0]:
clean_df = clean_df.drop(
    "row_num",
    "salary_error",
    "status_error",
    "country_error",
    "email_error",
    "hire_date_error",
    "error_reasons"
    
)

In [0]:
from pyspark.sql.functions import to_date


clean_df = clean_df.withColumn(
    "hire_date",
    to_date(col("hire_date_parsed")
))

In [0]:
clean_df.select(
    "hire_date",
    "hire_date_parsed"
).printSchema()

In [0]:
clean_df.printSchema()

In [0]:
clean_df = clean_df.drop(
    "hire_date_parsed"
)

In [0]:
clean_df.printSchema()

In [0]:
print(clean_df.count())

## Write Silver Tables

In [0]:
(
clean_df.write
.format("delta")
.mode("overwrite")
.saveAsTable("employees_clean")
)

In [0]:
%sql
select *
from employees_clean

In [0]:
%sql
describe employees_clean

In [0]:
display(rejected_df)

In [0]:
rejected_df = rejected_df.drop(
    "row_num"
)

In [0]:
rejected_df.printSchema()

In [0]:
(
rejected_df.write
.format("delta")
.mode("overwrite")
.saveAsTable("employees_rejected")
)

In [0]:
%sql

SELECT COUNT(*) AS row_count
FROM employees_rejected

In [0]:
%sql

DESCRIBE employees_rejected

In [0]:
display(excluded_df)

In [0]:
excluded_df = excluded_df.withColumn(
    "exclusion_reason",
    lit("OLDER_DUPLICATE")
)

In [0]:
excluded_df.select(
    "employee_id",
    "row_num",
    "exclusion_reason"
).show()

In [0]:
(
excluded_df.write
.format("delta")
.mode("overwrite")
.saveAsTable("employees_excluded")
)

#**Final Validation**

In [0]:


%sql

SELECT
    (SELECT COUNT(*) FROM employees_clean) AS clean_count,
    (SELECT COUNT(*) FROM employees_rejected) AS rejected_count,
    (SELECT COUNT(*) FROM employees_excluded) AS excluded_count,
    
    (
        (SELECT COUNT(*) FROM employees_clean)
        +
        (SELECT COUNT(*) FROM employees_rejected)
        +
        (SELECT COUNT(*) FROM employees_excluded)
    ) AS total_count